# Import functions

In [2]:
%run "..\Model\DataHelpers.ipynb"

import os
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
import numpy as np
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.metrics import auc

# LIME
from lime.lime_text import LimeTextExplainer
import lime.lime_tabular

class_names = ['nTNBC', 'TNBC']

targetGenes = ['CD1A', 'CSF2RB', 'EPCAM', 'ERBB2', 'ESR1', 'EZH2', 'FGB', 'FOXA1',
       'FOXC1', 'GATA3', 'LAMA2', 'LMNA', 'MDGA2', 'OBSCN', 'OGN', 'PGR',
       'SELL', 'SRC', 'TACSTD2', 'TBC1D22B', 'TFF1', 'TGFB3', 'UBE2C', 'VTCN1',
       'WTAP', 'YES1', 'YOD1']

# Load Dataset

In [3]:
import pandas as pd

RANDOM_STATE = 42
# patient_genes_literature
FILE_PATH = "../Data/patient_genes_lasso.csv" # Can be replaced with desired variant for different feature sets
variant = 'svm'
df = pd.read_csv(FILE_PATH)

# Split DataSet

In [4]:
X, y, X_train, X_test, y_train, y_test, test_case_id = split_data(df, "tnbc", "case_id")

X_train.shape=(781, 27)
X_test.shape=(196, 27)
y_train.shape=(781,)
y_test.shape=(196,)


# Train model and apply LIME

In [11]:
for modelName in ModelVariant.__members__:
    print(f'*** - Applying {modelName} to LIME - Start')
    model = getModel(modelName)

    # Train the model
    model.fit(X_train.to_numpy(), y_train)

    # Convert to numpy arrays
    X_train_array = X_train.to_numpy()
    X_test_array = X_test.to_numpy()

    #########################
    # LIME
    #
    # Get explainer
    #########################
    explainer = lime.lime_tabular.LimeTabularExplainer(X_train_array,
                                                       training_labels=y_train,
                                                       feature_names=targetGenes,
                                                       class_names=['nTNBC', 'TNBC'], 
                                                       categorical_features=[],
                                                       verbose=False,
                                                       mode='classification')

    #########################
    # LIME
    #
    # Get explanations 
    #########################
    results = dict()

    for i in range(len(y_test)):
        explanation = explainer.explain_instance(np.array(X_test.iloc[i]), model.predict_proba, num_features=27)
        
        # Use the feature as a key, add weight as value
        valuesOfExplanation = explanation.as_map()[1] # key is always 1
    
        results[i] = {'features': valuesOfExplanation, 'probability' : explanation.predict_proba, 'actual': y_test.iloc[i]}
    
    # results = dict()
    
    # for i in range(len(X_test_array)):
    #     explanation = explainer.explain_instance(X_test_array[i], model.predict_proba, num_features=27)
        
    #     # Use the feature as a key, add weight as value
    #     valuesOfExplanation = explanation.as_map()[1] # key is always 1
    
    #     # loop over each feature
    #     for v in valuesOfExplanation:
    #         if i == 0:
    #             results[v[0]] = [v[1]]
    #         else:
    #             results[v[0]].append(v[1])

    #########################
    # Get model tn/fn/tp/fp
    #########################
    y_pred = model.predict(X_test_array)
    y_testDf = pd.DataFrame(np.array(y_test))
    y_predDf = pd.DataFrame(y_pred)
    y_testDf.rename(columns={0: 'real'}, inplace=True)
    y_predDf.rename(columns={0: 'model'}, inplace=True)
    preds = pd.concat([y_testDf, y_predDf], axis=1)

    #########################
    # LIME AGGREGATES
    #########################
    falseNegative = preds[(preds['real'] != preds['model']) & (preds['real'] == True)]
    falsePositive = preds[(preds['real'] != preds['model']) & (preds['real'] == False)]
    truePositive  = preds[(preds['real'] == preds['model']) & (preds['real'] == True)]
    trueNegative  = preds[(preds['real'] == preds['model']) & (preds['real'] == False)]

    aggregateFalseNegative = dict()
    aggregateFalsePositive = dict()
    aggregateTruePositive  = dict()
    aggregateTrueNegative  = dict()
    
    first = True
    for i in falseNegative.index.tolist():
        # loop over each feature
        for v in results[i]['features']:
            if first:
                aggregateFalseNegative[v[0]] = [v[1]]
            else:
                aggregateFalseNegative[v[0]].append(v[1])
        first = False
    
    first = True
    for i in falsePositive.index.tolist():
        # loop over each feature
        for v in results[i]['features']:
            if first:
                aggregateFalsePositive[v[0]] = [v[1]]
            else:
                aggregateFalsePositive[v[0]].append(v[1])
        first = False
    
    
    first = True
    for i in truePositive.index.tolist():
        # loop over each feature
        for v in results[i]['features']:
            if first:
                aggregateTruePositive[v[0]] = [v[1]]
            else:
                aggregateTruePositive[v[0]].append(v[1])
        first = False
    
    first = True
    for i in trueNegative.index.tolist():
        # loop over each feature
        for v in results[i]['features']:
            if first:
                aggregateTrueNegative[v[0]] = [v[1]]
            else:
                aggregateTrueNegative[v[0]].append(v[1])
        first = False
    
    #########################
    # LIME
    #
    # Get columnnames
    # Convert dictionary to dataframe
    #########################
    colDict = dict()

    for i in range(len(explainer.feature_names)):
        colDict[i] = explainer.feature_names[i]


 
 
    print('***********************************')
    print(f'MODEL: {modelName}')
    print('***********************************')
    # Results
    print(f'True Negative  - count {len(trueNegative)} - {pd.DataFrame.from_dict(aggregateTrueNegative).rename(columns=colDict).describe().T['mean']}')
    print(f'True Positive  - count {len(truePositive)} - {pd.DataFrame.from_dict(aggregateTruePositive).rename(columns=colDict).describe().T['mean']}')
    print(f'False Negative - count {len(falseNegative)} - {pd.DataFrame.from_dict(aggregateFalseNegative).rename(columns=colDict).describe().T['mean']}')
    print(f'False Positive - count {len(falsePositive)} - {pd.DataFrame.from_dict(aggregateFalseNegative).rename(columns=colDict).describe().T['mean']}')
    print('***********************************')

    print(f'*** - Applying {modelName} to LIME - End')

*** - Applying SVM to LIME - Start
***********************************
MODEL: SVM
***********************************
True Negative  - count 166 - CD1A        2.686536e-04
PGR        -1.820067e-03
ESR1       -4.966809e-03
TGFB3      -2.445628e-03
EPCAM      -1.760626e-04
SRC        -7.527766e-05
TBC1D22B   -1.955210e-04
TFF1       -2.422060e-03
ERBB2      -3.972257e-03
UBE2C      -1.481257e-04
VTCN1      -2.159969e-04
GATA3      -3.447748e-03
LAMA2      -1.208651e-03
MDGA2      -9.468456e-04
EZH2       -3.287741e-04
FOXA1      -2.209501e-04
LMNA       -9.041096e-07
FOXC1      -1.384440e-03
SELL       -1.084843e-03
FGB        -4.450542e-04
TACSTD2     4.824245e-04
OGN        -4.867016e-04
OBSCN       1.928427e-04
YES1       -3.518837e-04
WTAP       -6.558578e-05
YOD1       -4.888723e-04
CSF2RB     -6.624719e-04
Name: mean, dtype: float64
True Positive  - count 20 - ESR1        0.041393
ERBB2       0.019337
TFF1        0.021049
GATA3       0.017525
TGFB3       0.011803
PGR         0.0115